In [1]:
import pandas as pd

In [10]:
battery_storage_capacity = { # in MWh
    'BALB1': 30,
    'BULBES1': 34,
    'GANNB1': 50,
    'HBESS1': 162,
    'KESSB1': 370,
    'PIBESS1': 10,
    'RANGEB1': 400,
    'VBB1': 470
}
round_trip_efficiency = 0.8

In [3]:
%%time
battery_df = pd.DataFrame()
for month in range(1,7):
    temp_dispatchload = pd.read_csv(f'/Volumes/EnergyData/AER/Raw_data_zipped/DISPATCHLOAD/PUBLIC_DVD_DISPATCHLOAD_2025{month:02d}010000.zip',
                                    skiprows=1,
                                    usecols=['SETTLEMENTDATE','DUID','INITIALMW','INITIAL_ENERGY_STORAGE','INTERVENTION'])\
                                    .dropna(subset=['DUID'])\
                                    .sort_values(by=['SETTLEMENTDATE','DUID','INTERVENTION'])\
                                    .drop_duplicates(subset=['SETTLEMENTDATE','DUID'], keep='last')\
                                    .drop(columns=['INTERVENTION'])\
                                    .reset_index(drop=True)
    temp_dispatchload = temp_dispatchload[temp_dispatchload['DUID'].isin(battery_storage_capacity.keys())].reset_index(drop=True)
    temp_dispatchload['SETTLEMENTDATE'] = pd.to_datetime(temp_dispatchload['SETTLEMENTDATE'])
    battery_df = pd.concat([battery_df, temp_dispatchload])
battery_df = battery_df.sort_values(by=['DUID','SETTLEMENTDATE'])\
                       .drop_duplicates(subset=['DUID','SETTLEMENTDATE'])\
                       .reset_index(drop=True)
battery_df['Storage_Change_MWh'] = battery_df.groupby('DUID')['INITIAL_ENERGY_STORAGE'].diff().fillna(0)
battery_df['Charge_or_Discharge_MWh'] = battery_df['Storage_Change_MWh']\
                                .apply(lambda x: x*(1/round_trip_efficiency) if x>0 else x) # account for round-trip efficiency

CPU times: user 35.9 s, sys: 1.92 s, total: 37.8 s
Wall time: 38 s


In [4]:
battery_df

,SETTLEMENTDATE,DUID,INITIALMW,INITIAL_ENERGY_STORAGE,Storage_Change_MWh,Charge_or_Discharge_MWh
0,2025-01-01 00:05:00,BALB1,0.0,10.74098,0.00000,0.00000
1,2025-01-01 00:10:00,BALB1,0.0,10.74098,0.00000,0.00000
2,2025-01-01 00:15:00,BALB1,0.0,10.73098,-0.01000,-0.01000
3,2025-01-01 00:20:00,BALB1,0.0,10.72098,-0.01000,-0.01000
4,2025-01-01 00:25:00,BALB1,4.9,10.41095,-0.31003,-0.31003
...,...,...,...,...,...,...
408955,2025-06-30 23:40:00,VBB1,41.2,68.50000,-1.40000,-1.40000
408956,2025-06-30 23:45:00,VBB1,13.2,65.80000,-2.70000,-2.70000
408957,2025-06-30 23:50:00,VBB1,-38.3,66.00000,0.20000,0.25000
408958,2025-06-30 23:55:00,VBB1,-18.7,68.00000,2.00000,2.50000


In [5]:
battery_df.groupby('DUID')['INITIAL_ENERGY_STORAGE'].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
DUID,,,,,,,,
BALB1,52122.0,8.31,5.47,0.0,4.33,6.41,10.88,26.47
BULBES1,52106.0,11.57,9.70,0.0,1.91,9.52,22.23,27.25
GANNB1,52015.0,21.87,8.88,0.0,13.70,21.80,30.70,36.50
HBESS1,52123.0,86.40,53.13,0.0,30.20,88.90,147.50,150.00
KESSB1,42958.0,93.04,139.90,0.0,0.00,0.00,165.47,387.80
PIBESS1,52114.0,5.47,3.55,0.0,2.03,5.05,9.04,11.15
RANGEB1,52124.0,181.84,159.37,4.9,34.40,117.90,354.20,451.00
VBB1,52090.0,249.03,114.08,0.0,172.90,258.40,344.60,461.40


In [ ]:
# total charging MWh per battery
round(battery_df[battery_df['Charge_or_Discharge_MWh']>0].groupby('DUID')['Charge_or_Discharge_MWh'].sum()) 

DUID
BALB1        4387.0
BULBES1      6857.0
GANNB1       7302.0
HBESS1      31707.0
KESSB1      25844.0
PIBESS1      2943.0
RANGEB1    104767.0
VBB1        79573.0
Name: Charge_or_Discharge_MWh, dtype: float64

In [ ]:
# battery storage capacity in MWh
pd.Series(battery_storage_capacity)

BALB1       30
BULBES1     34
GANNB1      50
HBESS1     162
KESSB1     370
PIBESS1     10
RANGEB1    400
VBB1       470
dtype: int64

In [ ]:
# charging cycles per battery
round(battery_df[battery_df['Charge_or_Discharge_MWh']>0].groupby('DUID')['Charge_or_Discharge_MWh'].sum()/pd.Series(battery_storage_capacity))

DUID
BALB1      146.0
BULBES1    202.0
GANNB1     146.0
HBESS1     196.0
KESSB1      70.0
PIBESS1    294.0
RANGEB1    262.0
VBB1       169.0
dtype: float64

In [13]:
battery_df.groupby('DUID')['INITIAL_ENERGY_STORAGE'].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
DUID,,,,,,,,
BALB1,52122.0,8.31,5.47,0.0,4.33,6.41,10.88,26.47
BULBES1,52106.0,11.57,9.70,0.0,1.91,9.52,22.23,27.25
GANNB1,52015.0,21.87,8.88,0.0,13.70,21.80,30.70,36.50
HBESS1,52123.0,86.40,53.13,0.0,30.20,88.90,147.50,150.00
KESSB1,42958.0,93.04,139.90,0.0,0.00,0.00,165.47,387.80
PIBESS1,52114.0,5.47,3.55,0.0,2.03,5.05,9.04,11.15
RANGEB1,52124.0,181.84,159.37,4.9,34.40,117.90,354.20,451.00
VBB1,52090.0,249.03,114.08,0.0,172.90,258.40,344.60,461.40
